In [20]:
import pandas as pd
import numpy as np
import random

In [21]:
np.random.seed(42)
random.seed(42)


In [22]:
#Load dataset

data = pd.read_csv('final_data.csv')

In [23]:
def spike(x, k, sigma):
    return x + k*sigma

def dip(x,k,sigma):
    return x - k*sigma
    

In [24]:
# Filter data from InnenStadt:

innenstadt = data[data['name'] == 'Innenstadt']


In [25]:
timestamps = list(innenstadt['timestamp'])

In [26]:
random.shuffle(timestamps)

In [27]:
timestamp_spike = timestamps[:20]
timestamp_dip = timestamps[20:40]
remaining_timestamp = [timestamp for timestamp in timestamps if (timestamp not in timestamp_spike and timestamp not in timestamp_dip)]

In [28]:
std = np.std(innenstadt['visitors'])

In [29]:
spike_idx = innenstadt['timestamp'].isin(timestamp_spike)
dip_idx = innenstadt['timestamp'].isin(timestamp_dip)
remaining_idx = innenstadt['timestamp'].isin(remaining_timestamp)


In [30]:
innenstadt.loc[spike_idx, 'injected_value'] = innenstadt.loc[spike_idx, 'visitors'].apply(
    lambda x: spike(x, 2, std)
)
innenstadt.loc[dip_idx, 'injected_value'] = (
    innenstadt.loc[dip_idx, 'visitors'] - 2 * std
).clip(lower=0)
innenstadt.loc[remaining_idx,'injected_value'] = (innenstadt.loc[remaining_idx,'visitors'] .apply(
    lambda x: x
))


In [31]:
innenstadt.loc[spike_idx, 'anomaly'] = 1
innenstadt.loc[dip_idx,'anomaly'] = 1
innenstadt.loc[remaining_idx,'anomaly'] = 0

In [32]:
innenstadt.to_csv('injected_data.csv')